In [ ]:
#Imports & Environment Setup

import os
import sys
import json
import time
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
import nest_asyncio

# Enable nested asyncio
nest_asyncio.apply()

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key 
groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or GROQ_API_KEY to .env"
    )

provider = "groq"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: groq
📁 Project root: c:\Users\viraj\Zuu\Real_State_Agent


In [2]:
# Load Configuration
from context_engineering.config import (
    validate, dump, CRAWL_OUT_DIR, MARKDOWN_DIR
)

# Validate and display config
try:
    validate()
    dump()
except Exception as e:
    print(f"⚠️  Config note: {e}")

# Ensure directories exist
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Output directories ready:")
print(f"   - Markdown: {MARKDOWN_DIR}")
print(f"   - JSONL: {CRAWL_OUT_DIR}")


CONFIGURATION (NON-SECRETS ONLY)

🌐 Provider:
   Provider: groq
   Model Tier: general
   Chat Model: llama-3.1-8b-instant
   Embedding Model: openai/text-embedding-3-large

📁 Directories:
   Data Root: c:\Users\viraj\Zuu\Real_State_Agent\data
   Vector Store: c:\Users\viraj\Zuu\Real_State_Agent\data\vectorstore
   Markdown: c:\Users\viraj\Zuu\Real_State_Agent\data\primelands_markdown
   Cache: c:\Users\viraj\Zuu\Real_State_Agent\data\cag_cache

🔧 Chunking:
   Fixed Size: 800 tokens
   Fixed Overlap: 100 tokens
   Sliding Window: 512 tokens
   Sliding Stride: 256 tokens
   Parent-Child: 250 → 1200 tokens
   Late Chunk: 1000 → 300 tokens

🔍 Retrieval:
   Top-K Results: 4
   Similarity Threshold: 0.7

💾 CAG:
   Cache TTL: 86400s
   Max Cache Size: 1000

🎯 CRAG:
   Confidence Threshold: 0.6
   Expanded K: 8



✅ Output directories ready:
   - Markdown: c:\Users\viraj\Zuu\Real_State_Agent\data\primelands_markdown
   - JSONL: c:\Users\viraj\Zuu\Real_State_Agent\data


In [11]:
# Correct import path for PrimelandsWebCrawler
from context_engineering.application.ingest_document_service.web_crawler import PrimelandsWebCrawler

print("✅ PrimelandsWebCrawler loaded from service layer")
print("📍 Location: context_engineering.application.ingest_document_service.web_crawler")

✅ PrimelandsWebCrawler loaded from service layer
📍 Location: context_engineering.application.ingest_document_service.web_crawler


In [ ]:
# Crawl Configuration
BASE_URL = "https://www.primelands.lk/"

# Only crawl relevant real estate and info pages
START_PATHS = [
    "/en", "/land/en", "/apartment/ongoing/en", "/house/en ", "portfolio-property/en","/about-us", "/contact-us/en", "/news-events", "/faq",
    "/projects", "/offers", "/careers", "/testimonials", "/blog"
]

START_URLS = [BASE_URL.rstrip("/") + path for path in START_PATHS]

# Exclude patterns for non-content or sensitive pages
EXCLUDE_PATTERNS = [
    "/login", "/terms", "/privacy", "/admin", "/cart", "/user", "/account",
    "/images/", "/downloads/", "/media/", ".jpg", ".png", ".pdf", ".jpeg", ".svg"
]

# Use config defaults if available
try:
    from context_engineering.config import CRAWL_MAX_DEPTH, CRAWL_DELAY_SECONDS
    MAX_DEPTH = CRAWL_MAX_DEPTH
    REQUEST_DELAY = CRAWL_DELAY_SECONDS
except ImportError:
    MAX_DEPTH = 3
    REQUEST_DELAY = 2.0

JSONL_PATH = CRAWL_OUT_DIR / "primelands_docs.jsonl"

print(f"🌐 Crawl config:")
print(f"   - Start URLs: {len(START_URLS)}")
print(f"   - Max depth: {MAX_DEPTH}")
print(f"   - Request delay: {REQUEST_DELAY}s")
print(f"   - Exclude patterns: {EXCLUDE_PATTERNS}")

🌐 Crawl config:
   - Start URLs: 14
   - Max depth: 3
   - Request delay: 2.0s
   - Exclude patterns: ['/login', '/terms', '/privacy', '/admin', '/cart', '/user', '/account', '/images/', '/downloads/', '/media/', '.jpg', '.png', '.pdf', '.jpeg', '.svg']


In [16]:
# Run Crawl with Service
import time

# Initialize crawler service
crawler = PrimelandsWebCrawler(
    base_url=BASE_URL,
    max_depth=MAX_DEPTH,
    exclude_patterns=EXCLUDE_PATTERNS
)

start_time = time.time()
print(f"\n🚀 Starting crawl at {time.strftime('%H:%M:%S')}\n")
documents = crawler.crawl(START_URLS, request_delay=REQUEST_DELAY)

elapsed = time.time() - start_time
print(f"\n✅ Crawl complete in {elapsed:.1f}s")
print(f"📄 Documents collected: {len(documents)}")
print(f"🔗 URLs visited: {len(crawler.visited)}")

Task exception was never retrieved
future: <Task finished name='Task-124' coro=<Connection.run() done, defined at c:\Users\viraj\Zuu\Real_State_Agent\mp_02\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\viraj\anaconda3\Lib\asyncio\tasks.py", line 267, in __step
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\Users\viraj\Zuu\Real_State_Agent\mp_02\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "c:\Users\viraj\Zuu\Real_State_Agent\mp_02\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "c:\Users\viraj\Zuu\Real_State_Agent\mp_02\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\viraj\anaconda3\Lib\asyncio\subprocess.py", lin


🚀 Starting crawl at 20:38:01



NotImplementedError: 